# Day 10: Object Tracking Basics
Today we will cover:
- Optical Flow
- KLT (Kanade–Lucas–Tomasi) Tracker

## Why Object Tracking?
Object tracking allows us to follow moving targets across video frames. It is essential for applications like surveillance, sports analytics, and autonomous driving.

## Optical Flow
Optical flow estimates pixel motion between consecutive frames.
- **Dense Optical Flow**: Computes motion for all pixels.
- **Sparse Optical Flow**: Tracks selected keypoints.

## KLT Tracker
The Kanade–Lucas–Tomasi tracker uses sparse optical flow to track good features (corners).
- Pros: Efficient, widely used.
- Cons: Sensitive to occlusion and large motion.

In [2]:
import cv2
import numpy as np

# Load video
cap = cv2.VideoCapture('chaplin.mp4')  # replace with your video file
if not cap.isOpened():
    raise IOError("Cannot open video file")

# Parameters for ShiTomasi corner detection
feature_params = dict(maxCorners=100,
                      qualityLevel=0.3,
                      minDistance=7,
                      blockSize=7)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15,15),
                 maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Take first frame and find corners in it
ret, old_frame = cap.read()
if not ret:
    raise IOError("Cannot read first frame from video")

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Create mask image for drawing
mask = np.zeros_like(old_frame)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    if p1 is None:
        break

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw tracks
    for new, old in zip(good_new, good_old):
        a, b = new.ravel()
        c, d = old.ravel()

        # Cast to int for drawing
        a, b, c, d = int(a), int(b), int(c), int(d)

        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame = cv2.circle(frame, (a, b), 5, (0, 0, 255), -1)

    img = cv2.add(frame, mask)

    cv2.imshow('Tracking', img)
    if cv2.waitKey(30) & 0xFF == 27:  # ESC to exit
        break

    # Update previous frame and points
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

cap.release()
cv2.destroyAllWindows()

## Exercise: Track Moving Ball in Video
Run the above code on `ball.mp4`. Observe how the tracker follows the ball’s motion. Try adjusting parameters like `maxCorners` or `winSize`.

## Mini Project: Motion Tracker
We will build a simple motion tracker that highlights moving objects using dense optical flow.

In [4]:
cap = cv2.VideoCapture('chaplin.mp4')  # replace with your video file
ret, frame1 = cap.read()
prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
hsv = np.zeros_like(frame1)
hsv[...,1] = 255

while True:
    ret, frame2 = cap.read()
    if not ret:
        break
    next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(prvs, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    hsv[...,0] = ang*180/np.pi/2
    hsv[...,2] = cv2.normalize(mag,None,0,255,cv2.NORM_MINMAX)
    bgr = cv2.cvtColor(hsv,cv2.COLOR_HSV2BGR)
    cv2.imshow('Motion Tracker', bgr)
    if cv2.waitKey(30) & 0xFF == 27:
        break
    prvs = next

cap.release()
cv2.destroyAllWindows()